### Прогноз цен на недвижимость

Курсовой проект по машинному обучению (4 курс, 1 семестр).
Авторы: Эрнест и Иршат.
Датасет: объявления о продаже недвижимости с портала Zingat.
Цель: исследовать данные, подготовить признаки и обучить модели регрессии для прогнозирования стоимости объектов.


In [ ]:
# подключаем библиотеки для работы с данными, графиками и ML
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

## Этап 1. Исследование данных (EDA)

### 1.1 Знакомство с данными
Загружаем датасет, смотрим его размерность, первые, последние и случайные строки.


In [ ]:
# читаем датасет из папки data
df = pd.read_csv("../data/real_estate_data.csv", low_memory=False)
print("Размер датасета (строк, колонок):", df.shape)
# выводим первые 5 строк
df.head()

In [ ]:
# смотрим последние 5 строк
df.tail()

In [ ]:
# выводим одну случайную строку для проверки разнообразия данных
df.sample(1, random_state=42)

### Описание признаков датасета (по документации Zingat)

* **id**: уникальный идентификатор объявления
* **type**: тип объекта (Konut - жилье)
* **sub_type**: подтип жилья (Daire - квартира, Villa - вилла, Rezidans - резиденция)
* **start_date** / **end_date**: даты публикации и снятия с сайта
* **listing_type**: тип сделки (1 - продажа, 2 - аренда)
* **tom**: дней на рынке (time on market)
* **building_age**: возраст строения (0, 1, 2, 6-10 arasi и т.д.)
* **total_floor_count**: этажность здания
* **floor_no**: этаж квартиры
* **room_count**: число комнат (формат 2+1, 3+1)
* **size**: общая площадь в м2
* **address**: город / район / микрорайон
* **furnished**: меблировка (100% пропусков)
* **heating_type**: система отопления (Kombi, Merkezi Sistem и др.)
* **price**: цена объекта в турецких лирах (целевая переменная)
* **price_currency**: валюта (TRY)


In [ ]:
# типы данных и ненулевые значения
df.info()

### 1.2 Описательная статистика и анализ целевой переменной


In [ ]:
# базовые статистики числовых колонок: минимум, максимум, среднее, квантили
df.describe()

In [ ]:
# статистика категориальных признаков: количество уникальных и самый частый класс
df.describe(include='O')

In [ ]:
# проверяем скошенность (асимметрию) цены:
# положительный коэффициент говорит о сильном хвосте вправо (дорогие элитные объекты)
print("Коэффициент асимметрии цены:", df['price'].skew())
print("Асимметрия после log1p:", np.log1p(df['price'].dropna()).skew())

### 1.3 Визуальный анализ данных (5+ типов графиков)


In [ ]:
# 1. Гистограммы распределения всех числовых признаков
df.hist(figsize=(20, 10))
plt.show()

In [ ]:
# 2. Круговая диаграмма долей подтипов недвижимости
plt.figure(figsize=(8, 6))
df['sub_type'].value_counts().head(5).plot(kind='pie', autopct='%.2f', textprops={'color': 'black'})
plt.title('Доли подтипов недвижимости', fontsize=14)
plt.ylabel('')
plt.show()

Больше 85% всех объявлений в датасете - это квартиры (Daire).


In [ ]:
# 3. Boxplot: распределение цены по подтипам недвижимости
top_subtypes = df['sub_type'].value_counts().head(5).index
plt.figure(figsize=(12, 5))
sns.boxplot(data=df[df['sub_type'].isin(top_subtypes)], x='sub_type', y='price')
plt.title('Sub Type Vs Price', fontsize=15)
plt.xlabel('Тип жилья')
plt.ylabel('Цена (TRY)')
plt.ylim(0, 3000000)
plt.show()

Виллы и резиденции стоят ощутимо дороже типовых квартир.


In [ ]:
# 4. Столбчатая диаграмма средних цен по городам
df['city'] = df['address'].apply(lambda x: str(x).split('/')[0] if '/' in str(x) else 'Diger')
city_mean = df.groupby('city')['price'].mean().sort_values(ascending=False).head(8)

plt.figure(figsize=(12, 5))
sns.barplot(x=city_mean.index, y=city_mean.values)
plt.title('City Vs Average Price', fontsize=15)
plt.xlabel('Город')
plt.ylabel('Средняя цена (TRY)')
plt.show()

Стамбул и курортные города (Мугла, Анталья) лидируют по средней стоимости жилья.


In [ ]:
# 5. Scatterplot (точечная диаграмма): зависимость цены от площади
sample_data = df[(df['size'] >= 30) & (df['size'] <= 300) & (df['price'] <= 5000000)].sample(3000, random_state=42)

plt.figure(figsize=(10, 6))
sns.scatterplot(data=sample_data, x='size', y='price', alpha=0.3)
plt.title('Size Vs Price', fontsize=15)
plt.xlabel('Площадь (м2)')
plt.ylabel('Цена (TRY)')
plt.show()

Видна прямая зависимость: чем больше площадь квартиры, тем выше цена.


### 1.4 Анализ пропусков, дубликатов и аномалий


In [ ]:
# таблица пропусков по колонкам
null_info = pd.DataFrame({
    'Пропусков': df.isnull().sum(),
    'Процент': (df.isnull().sum() / len(df) * 100).round(2)
})
null_info[null_info['Пропусков'] > 0]

In [ ]:
# проверяем наличие полных дубликатов строк
print("Количество полных дубликатов:", df.duplicated().sum())

# проверяем нереалистичные значения (цена <= 0, площадь <= 0)
bad_prices = (df['price'] <= 0).sum()
bad_sizes = (df['size'] <= 0).sum()
print("Записей с нулевой или отрицательной ценой:", bad_prices)
print("Записей с нулевой или отрицательной площадью:", bad_sizes)

### 1.5 Промежуточные выводы и гипотезы EDA

На основе анализа формулируем 4 гипотезы для последующего моделирования:
1. **Гипотеза о площади**: площадь объекта (size) сильнее всего положительно коррелирует со стоимостью.
2. **Гипотеза о локации**: город и район являются вторым ключевым фактором цены (Стамбул дороже других регионов).
3. **Гипотеза о комнатах**: количество комнат (room_count) увеличивает цену объекта, так как растет полезная площадь.
4. **Гипотеза об отоплении**: наличие современного газового отопления (Kombi) повышает ликвидность и цену квартиры.
